# CSF-CUF · Interactive Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/quickstart_colab.ipynb)

**Run a CUF analysis without writing Python or editing YAML files.**

This notebook uses the original [CSF-CUF Quick Start](https://github.com/giovanniboscu/continuous-section-field/tree/main/cuf/quickstart): the CSF model (geometry/material), the problem (load/boundary conditions), and the CUF numerical case.

**First use:** select **Runtime → Run all**. Afterwards, change the parameters in step 2 and press ▶ on that cell. Results appear immediately below. No GitHub account or GitHub authorization is needed to run this public example.

The notebook downloads the public repository read-only and works on temporary YAML copies. It never pushes changes to GitHub.


### 1 · Prepare the environment
Run once per new Colab runtime. This clones the public repository and installs the solver and YAML support.

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

REPO = Path('/content/continuous-section-field')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/giovanniboscu/continuous-section-field.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'pypardiso', 'pyyaml'], check=True)
QUICKSTART = REPO / 'cuf' / 'quickstart'
assert (QUICKSTART / 'models/rectangular_prismatic_csf.yaml').is_file()
assert (QUICKSTART / 'problems/torsion_halfwave.yaml').is_file()
assert (QUICKSTART / 'cases/torsion_halfwave_legendre_N02.yaml').is_file()
print('CSF-CUF ready. Choose parameters below and press Run.')

### 2 · Configure and run

Change the values in the form and press **▶** on this cell. **S0** is the section at 0 mm; **S1** is at 1000 mm. Equal dimensions and equal moduli reproduce the original prismatic homogeneous model. Different values introduce longitudinal variation. Units: **mm**, **MPa**, **N**.

The initial values reproduce the original N02 case. The longitudinal order is independent of the transverse CUF order. For a first experiment, change only one value and rerun.

In [ ]:
# @title Configure model and run CSF-CUF { display-mode: "form" }
# @markdown **Numerical approximation**
cuf_order = 2 # @param {type:"integer"}
longitudinal_order = 6 # @param {type:"integer"}
# @markdown **Material (Young's modulus, MPa)**
E_S0 = 71700.0 # @param {type:"number"}
E_S1 = 71700.0 # @param {type:"number"}
# @markdown **Rectangular section dimensions (mm)**
width_S0 = 100.0 # @param {type:"number"}
height_S0 = 100.0 # @param {type:"number"}
width_S1 = 100.0 # @param {type:"number"}
height_S1 = 100.0 # @param {type:"number"}

import yaml
from IPython.display import display, Markdown

if not (isinstance(cuf_order, int) and 1 <= cuf_order <= 20):
    raise ValueError('CUF order must be an integer from 1 to 20.')
if not (isinstance(longitudinal_order, int) and 1 <= longitudinal_order <= 20):
    raise ValueError('Longitudinal order must be an integer from 1 to 20.')
for label, value in [('E_S0', E_S0), ('E_S1', E_S1),
                     ('width_S0', width_S0), ('height_S0', height_S0),
                     ('width_S1', width_S1), ('height_S1', height_S1)]:
    if not (0 < float(value) < float('inf')):
        raise ValueError(f'{label} must be positive and finite.')

# Work exclusively on disposable copies; preserve original relative YAML paths.
WORK = Path('/content/csf_cuf_quickstart_work')
for folder in ('models', 'problems', 'cases'):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
for folder, filename in (
    ('models', 'rectangular_prismatic_csf.yaml'),
    ('problems', 'torsion_halfwave.yaml'),
    ('cases', 'torsion_halfwave_legendre_N02.yaml'),
):
    shutil.copy2(QUICKSTART / folder / filename, WORK / folder / filename)

model_path = WORK / 'models/rectangular_prismatic_csf.yaml'
case_path = WORK / 'cases/torsion_halfwave_legendre_N02.yaml'
model = yaml.safe_load(model_path.read_text(encoding='utf-8'))
case = yaml.safe_load(case_path.read_text(encoding='utf-8'))

def rectangle(width, height):
    w, h = width / 2.0, height / 2.0
    return [[-w, -h], [w, -h], [w, h], [-w, h]]

for station, E, width, height in (
    ('S0', E_S0, width_S0, height_S0),
    ('S1', E_S1, width_S1, height_S1),
):
    polygons = model['CSF']['sections'][station]['polygons']
    assert len(polygons) == 1, f'Expected one rectangle in {station}'
    polygons[0]['weight'] = float(E)
    polygons[0]['vertices'] = rectangle(float(width), float(height))

case['cuf']['order'] = cuf_order
case['longitudinal']['order'] = longitudinal_order
model_path.write_text(yaml.safe_dump(model, sort_keys=False), encoding='utf-8')
case_path.write_text(yaml.safe_dump(case, sort_keys=False), encoding='utf-8')

print(f'Running CUF N{cuf_order:02d}; longitudinal order {longitudinal_order}')
print(f'E: S0={E_S0:g}, S1={E_S1:g} MPa')
print(f'Geometry: S0={width_S0:g} x {height_S0:g}, S1={width_S1:g} x {height_S1:g} mm')
print('Solving...')
run = subprocess.run(
    ['csf-cuf', 'cases/torsion_halfwave_legendre_N02.yaml'],
    cwd=WORK, capture_output=True, text=True,
)
if run.stdout.strip():
    print(run.stdout)
if run.returncode != 0:
    if run.stderr.strip():
        print(run.stderr)
    raise RuntimeError(f'csf-cuf failed (exit code {run.returncode})')
if run.stderr.strip():
    print(run.stderr)

RESULTS = WORK / 'output/torsion_halfwave_legendre_N02'
response_path = RESULTS / 'response.txt'
if not response_path.is_file():
    raise FileNotFoundError(f'Solver completed but response.txt was not found: {response_path}')
response_text = response_path.read_text(encoding='utf-8')
display(Markdown('### Structural response (`response.txt`)'))
print(response_text)

# Keep previous response in this runtime for a simple side-by-side comparison.
previous = globals().get('_previous_csf_response')
if previous is not None:
    display(Markdown('### Previous run (for comparison)'))
    print(previous)
_previous_csf_response = response_text
print('Results:', RESULTS)
print('Original GitHub YAML files remain unchanged.')

**Done.** Change one or more fields in step 2, then run that cell again. The previous response is shown below the current one. Original YAML files remain unchanged.

**For maintainers:** to make this notebook available through the public Colab URL, commit this file to `main/cuf/quickstart/quickstart_colab.ipynb`. The “Open in Colab” badge belongs in the GitHub README, not in this notebook.

**Scope:** didactic demonstration, not validation or convergence study. Low CUF orders may approximate torsion poorly. [Quick Start README](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/README.md).